In [1]:
# Install Ultralytics
!pip install ultralytics -q

import os
import cv2
import torch
import xml.etree.ElementTree as ET
from xml.dom import minidom
from collections import defaultdict
from ultralytics import YOLO

print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete.


In [2]:
# --- PATHS ---
# Path to your trained model
MODEL_PATH = "/kaggle/input/updated-yolov8n-model/pytorch/default/1/best.pt"

# Paths to the new videos
VIDEO_PATHS = [
    "/kaggle/input/bdd-ped2/NO20251114-155337-143771F.MP4",
    "/kaggle/input/bdd-ped2/NO20251114-155438-143772F.MP4"
]

# Output directory for the XML files
OUTPUT_DIR = "/kaggle/working/annotations"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- CLASS MAPPING ---
# These must match the order used during your training
CLASSES = [
    "person",       # 0
    "car",          # 1
    "rickshaw",     # 2
    "cng",          # 3
    "bus",          # 4
    "truck",        # 5
    "motorbike",    # 6
    "bicycle",      # 7
    "leguna",       # 8
    "bangla-tesla"  # 9
]

print(f"Model: {MODEL_PATH}")
print(f"Processing {len(VIDEO_PATHS)} videos...")

Model: /kaggle/input/updated-yolov8n-model/pytorch/default/1/best.pt
Processing 2 videos...


In [3]:
def create_cvat_xml(video_name, width, height, num_frames, tracks, output_path):
    """
    Generates a CVAT 1.1 Video XML file from tracking data.
    """
    root = ET.Element("annotations")
    
    # Metadata section
    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "size").text = str(num_frames)
    ET.SubElement(task, "mode").text = "interpolation"
    ET.SubElement(task, "overlap").text = "0"
    
    # Source section
    source = ET.SubElement(meta, "source")
    ET.SubElement(source, "original_size").text = f"{width}x{height}"
    
    # Tracks section
    # tracks dictionary format: {track_id: [{'frame': int, 'bbox': [xtl, ytl, xbr, ybr], 'label': str}, ...]}
    
    for track_id, detections in tracks.items():
        # Get label from the first detection in the track (assuming consistent)
        label_name = detections[0]['label']
        
        # Create track element
        track_elem = ET.SubElement(root, "track")
        track_elem.set("id", str(track_id))
        track_elem.set("label", label_name)
        
        for det in detections:
            box = ET.SubElement(track_elem, "box")
            box.set("frame", str(det['frame']))
            box.set("outside", "0")
            box.set("occluded", "0")
            box.set("keyframe", "1")
            
            # Coordinates
            xtl, ytl, xbr, ybr = det['bbox']
            
            # Clamp coordinates to image boundaries
            xtl = max(0, min(width, xtl))
            ytl = max(0, min(height, ytl))
            xbr = max(0, min(width, xbr))
            ybr = max(0, min(height, ybr))
            
            box.set("xtl", f"{xtl:.2f}")
            box.set("ytl", f"{ytl:.2f}")
            box.set("xbr", f"{xbr:.2f}")
            box.set("ybr", f"{ybr:.2f}")
            
    # Save formatted XML
    xmlstr = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
    with open(output_path, "w") as f:
        f.write(xmlstr)
    
    print(f"Saved: {output_path}")

In [4]:
# Load the model
model = YOLO(MODEL_PATH)

for video_path in VIDEO_PATHS:
    if not os.path.exists(video_path):
        print(f"Warning: Video not found at {video_path}")
        continue
        
    video_basename = os.path.basename(video_path)
    xml_filename = os.path.splitext(video_basename)[0] + ".xml"
    save_path = os.path.join(OUTPUT_DIR, xml_filename)
    
    print(f"\nProcessing: {video_basename}...")
    
    # Open Video to get properties
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    # Run Inference with Tracking
    # stream=True reduces memory usage
    # persist=True ensures IDs persist between frames
    results = model.track(source=video_path, stream=True, persist=True, conf=0.4, verbose=False)
    
    # Store data: { track_id: [ {frame, bbox, label}, ... ] }
    all_tracks = defaultdict(list)
    
    frame_idx = 0
    
    for result in results:
        # Check if any detections exist in this frame
        if result.boxes is not None and result.boxes.id is not None:
            
            boxes = result.boxes.xyxy.cpu().numpy() # x1, y1, x2, y2
            ids = result.boxes.id.cpu().numpy()     # Track IDs
            cls = result.boxes.cls.cpu().numpy()    # Class IDs
            
            for box, track_id, class_id in zip(boxes, ids, cls):
                class_name = CLASSES[int(class_id)]
                
                # Append to our track storage
                all_tracks[int(track_id)].append({
                    'frame': frame_idx,
                    'bbox': box,
                    'label': class_name
                })
        
        frame_idx += 1
        if frame_idx % 100 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames...", end='\r')
            
    # Generate the XML
    create_cvat_xml(video_basename, width, height, frame_idx, all_tracks, save_path)

print("\n\nAll videos processed!")


Processing: NO20251114-155337-143771F.MP4...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 314ms
Prepared 1 package in 79ms
Installed 1 package in 3ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Saved: /kaggle/working/annotations/NO20251114-155337-143771F.xml

Processing: NO20251114-155438-143772F.MP4...
Saved: /kaggle/working/annotations/NO20251114-155438-143772F.xml


All videos processed!


In [5]:
import shutil
from IPython.display import FileLink

# Zip the annotations folder
shutil.make_archive("/kaggle/working/automated_annotations", 'zip', OUTPUT_DIR)

print("Annotation complete! Download your XML files here:")
display(FileLink("automated_annotations.zip"))

Annotation complete! Download your XML files here:


/kaggle/working/automated_annotations.zip

## Demo

In [6]:
import cv2
import os
import xml.etree.ElementTree as ET
import random
from collections import defaultdict
from IPython.display import FileLink

# Initialize a fixed color map for consistency
# You can customize these colors (B, G, R)
COLORS = {
    "person": (0, 0, 255),       # Red
    "car": (255, 0, 0),          # Blue
    "rickshaw": (0, 255, 255),   # Yellow
    "cng": (0, 255, 0),          # Green
    "bus": (255, 0, 255),        # Magenta
    "truck": (128, 0, 128),      # Purple
    "motorbike": (255, 165, 0),  # Orange
    "bicycle": (0, 128, 255),    # Light Orange
    "leguna": (128, 128, 0),     # Teal
    "bangla-tesla": (255, 255, 0) # Cyan
}

def get_color(label):
    if label in COLORS:
        return COLORS[label]
    return (255, 255, 255) # White for unknown

print("Libraries imported.")

Libraries imported.


In [7]:
# --- PATHS ---
VIDEO_PATH = "/kaggle/input/bdd-ped2/NO20251114-155337-143771F.MP4"
XML_PATH = "/kaggle/input/updated-annotation/NO20251114-155337-143771F.xml"

# Output filename
OUTPUT_VIDEO_NAME = "annotated_demo.mp4"
OUTPUT_PATH = f"/kaggle/working/{OUTPUT_VIDEO_NAME}"

print(f"Input Video: {VIDEO_PATH}")
print(f"Input XML: {XML_PATH}")
print(f"Output Target: {OUTPUT_PATH}")

Input Video: /kaggle/input/bdd-ped2/NO20251114-155337-143771F.MP4
Input XML: /kaggle/input/updated-annotation/NO20251114-155337-143771F.xml
Output Target: /kaggle/working/annotated_demo.mp4


In [8]:
def parse_xml_annotations(xml_file):
    """
    Parses CVAT 1.1 Video XML and returns a dictionary:
    { frame_number: [ (label, track_id, x1, y1, x2, y2), ... ] }
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Dictionary to store boxes per frame
    # Key: frame_id, Value: list of detections
    frame_data = defaultdict(list)
    
    # Iterate through all 'track' elements
    for track in root.findall('track'):
        label = track.get('label')
        track_id = track.get('id')
        
        # Iterate through all 'box' elements inside the track
        for box in track.findall('box'):
            frame = int(box.get('frame'))
            xtl = float(box.get('xtl'))
            ytl = float(box.get('ytl'))
            xbr = float(box.get('xbr'))
            ybr = float(box.get('ybr'))
            
            # Store data
            frame_data[frame].append({
                "label": label,
                "id": track_id,
                "bbox": (int(xtl), int(ytl), int(xbr), int(ybr))
            })
            
    return frame_data

print("Parser defined.")

Parser defined.


In [9]:
# 1. Parse the Annotations
print("Parsing XML annotations...")
annotations = parse_xml_annotations(XML_PATH)
print(f"Loaded annotations for {len(annotations)} frames.")

# 2. Open Video Source
cap = cv2.VideoCapture(VIDEO_PATH)

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video Info: {width}x{height} @ {fps} FPS, {total_frames} frames.")

# 3. Setup Video Writer
# 'mp4v' is a standard codec that works well in most environments
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_idx = 0

print("Starting video rendering...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Check if this frame has annotations
    if frame_idx in annotations:
        for obj in annotations[frame_idx]:
            label = obj['label']
            track_id = obj['id']
            x1, y1, x2, y2 = obj['bbox']
            
            # Get color
            color = get_color(label)
            
            # Draw Rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Draw Label Background
            label_text = f"{label} ID:{track_id}"
            (text_w, text_h), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
            
            # Draw Text
            cv2.putText(frame, label_text, (x1, y1 - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Write frame to output video
    out.write(frame)
    
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"Processed {frame_idx}/{total_frames} frames...", end='\r')

# Cleanup
cap.release()
out.release()
print(f"\nDone! Video saved to: {OUTPUT_PATH}")

Parsing XML annotations...
Loaded annotations for 1796 frames.
Video Info: 2592x1944 @ 30.0 FPS, 1800 frames.
Starting video rendering...
Processed 1800/1800 frames...
Done! Video saved to: /kaggle/working/annotated_demo.mp4


In [10]:
print("Click below to download your annotated video:")
display(FileLink(OUTPUT_VIDEO_NAME))

Click below to download your annotated video:


/kaggle/working/annotated_demo.mp4